# HOOMD-blue, MD and Activity 

---

### What is MD? 

- MD is a computer simulation method for investigating systems of particles or atoms
- Very simply put, from each configuration, MD evaluates the forces acting on each particle and moves them according to Newton's equation of motion for a small timestep 
- unsurprisingly this process is repeated many thousands, millions or billions of times to produce a 'trajectory' of the system
- Here we will look at Brownian MD (which involves making some, good, approximations of Newton's equations to accelerate the process) where the equation of motion for each particle is:

$$\dot{\mathbf{r}}_i = \frac{1}{\gamma}\mathbf{F}_i^{\text{Ext}} + \sqrt{2D_t}\,\boldsymbol{\xi}_i$$

- Where $r_i$ is the position of particle $i$, $\gamma$ is the friction of the medium in which the particles evolve, $ \mathbf{F}_i^{\text{Ext}}$ is the forces acting on the particle, this can come from other particles, external potentials, activity (we will talk about later), and a variety of other sources (which we, in all likelihood, will not involve in our simulations). 
- The final term in our Brownian equation of motion is what introduces randomness into our otherwise deterministic equations. $\boldsymbol{\xi}_i$ is a random force on particle $i$ which is 'delta-correlated', i.e. from timestep to timestep, particle to particle, and direction (x, y, z), is uncorrelated. Each Cartesian component of $\boldsymbol{\xi}_i$ is drawn independently from a Gaussian distribution centred on $0$ with variance $1$; $D_t$ is the translational diffusion constant and controls the size of these random thermal kicks.
- We can relate $\gamma$ and $D_t$ through the temperature: $D_t=\frac{k_BT}{\gamma}$ i.e. the higher the temperature, the bigger the thermal kicks 
- To learn more about MD, the best thing I have read is: Frenkel and Smit, UnderstandingMolecular Simualtion, Chapter 4 on MD simaultions. This will run you though the basic algorithms going on under the hood

--- 

### Activity 

- In the context of soft matter physics, active systems involve the consumption of an energy store which changes the behaviour of some or all constituents. This can involve changing shape, size or propelling themselves through their medium. 
- The latter is what we are interested in. We can describe activity of this kind by simply adding one more term to our equation of motion:
$$    \dot{\mathbf{r}}_i = \frac{1}{\gamma}\mathbf{F}_i^{\text{Ext}} + \sqrt{2D_t}\,\boldsymbol{\xi}_i + \frac{1}{\gamma} F_0 \hat{\mathbf{e}}_i$$
- $\hat{\mathbf{e}}_i$ is the direction of propulsion and $F_0$ is the force with which it propels itself. 
- $\hat{\mathbf{e}}_i$ is subject to rotational diffusion i.e. the direction the particle propels itself changes through time
- specifically this is characterised by the following equation: $\dot{\hat{\mathbf{e}}}_i = \sqrt{2D_r}\,\boldsymbol{\eta}_i \times \hat{\mathbf{e}}_i$
- Similarly to $\boldsymbol{\xi}_i$, $\boldsymbol{\eta}_i$ is delta correlated which means that the correlation between $\hat{\mathbf{e}}_i(t)$ and $\hat{\mathbf{e}}_i(t+\Delta t)$ decays exponentially with respect to $\Delta t$

---

### What is HOOMD-blue?

- HOOMD-blue is a python package which runs, among other things, molecular dynamics (MD) simulation 
- We set up the simulation in python, all the parameters and starting configurations of the system etc. 
- We send the starting configuration, parameters and rules for evolving the system 
- The workhorse of our simulations, HOOMD-blue, uses C++ and CUDA (If we are using GPUs) to run the simulations 

--- 

### Building HOOMD

- HOOMD is not trivial to install on your PC (We can have a look at what OS you have and that will change it)
- For work on the cluster things are actually simpler, I already have it compiled so I can just send to you
- Once we have it built we can give some simple simulations a go



First we import HOOMD and some other stuff

In [ ]:
import hoomd # simulation engine
import numpy as np # arrays and random numbers
import gsd.hoomd # reading and writing HOOMD trajectory files
import os # deleting old files

### Units
- We use reduced units: lengths in particle diameters $\sigma$, energies in $k_BT$, and time in $\tau = \gamma\sigma^2/k_BT$ (roughly the time a particle takes to diffuse its own size)
- So `L = 100` is 100 particle diameters and `dt = 1e-4` is $10^{-4}\tau$

In [ ]:
L = 100 # size of our 2D box
A = L**2 # Area of our 2D box
N = 100 # number of particles
kT = 1.0 # temperature (thermal energy)

### Periodic boundaries
- The box has no walls: a particle leaving one side comes back in on the opposite side, so a small box behaves like a piece of a much bigger system
- Distances between particles are measured the short way round the box (the `np.round` line below)

In [19]:
# place N particles at random, rejecting any within 1.2 sigma of another so none overlap
rng = np.random.default_rng() # random number generator
positions = np.zeros((N, 3)) # x, y, z for each particle (z stays 0 in 2D)
placed = 0 # number of particles placed so far
while placed < N:
    trial = (rng.random(2) - 0.5) * L # random x, y in [-L/2, L/2)
    d = positions[:placed, :2] - trial # separation from each placed particle
    d -= L * np.round(d / L)  # periodic boundaries
    if placed == 0 or np.min(np.sum(d**2, axis=1)) > 1.2**2: # accept if nothing is closer than 1.2
        positions[placed, :2] = trial
        placed += 1


In [20]:
# build the initial frame to save to file
frame = gsd.hoomd.Frame() 
frame.particles.N = N 
frame.particles.position = positions
frame.configuration.box = [L, L, 0, 0, 0, 0] # Lx, Ly, Lz, xy, xz, yz (Lz = 0 makes it 2D)

In [ ]:
# delete any old initial config file
init_filename="trajectory_init.gsd"
if os.path.exists(init_filename):
    os.remove(init_filename)
    
# save the initial config ("x" creates a new file)
with gsd.hoomd.open(name=init_filename, mode="x") as f:
    f.append(frame)

In [ ]:
import matplotlib.pyplot as plt # plotting

# read the initial config back from the file and plot it
with gsd.hoomd.open(name=init_filename, mode="r") as f:
    init = f[0]

Lx, Ly = init.configuration.box[:2] # box size
fig, ax = plt.subplots(figsize=(6, 6))
for x, y, z in init.particles.position:
    ax.add_patch(plt.Circle((x, y), 0.5)) # particle of diameter 1 (sigma), drawn to scale
ax.set_xlim(-Lx / 2, Lx / 2)
ax.set_ylim(-Ly / 2, Ly / 2)
ax.set_aspect("equal")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("Initial configuration")
plt.show()

### Random seeds
- The seed fixes the random kicks, so the same seed gives exactly the same run
- For real results, repeat runs with different seeds and average them (with error bars)

In [23]:
GPU = hoomd.device.GPU() #This is if you have a GPU we can use, otherwise use CPU() instead of GPU()
simulation = hoomd.Simulation(device=GPU, seed=1) # seed sets the random numbers
simulation.create_state_from_gsd(filename=init_filename) # load the initial config

### Timestep
- If `dt` is too big, particles can jump into each other in one step, the forces become huge and the simulation blows up
- `dt = 1e-4` is safe here; if you see NaN or particles flying off, make it smaller

In [24]:
integrator = hoomd.md.Integrator(dt=1e-4) # moves the system forward in time, dt is the timestep

### Why no mass?
- Small particles in a fluid live at low Reynolds number: drag dominates inertia, so they stop almost instantly when the force stops
- That's why the Brownian equation above has no mass or acceleration, and $D_t = k_BT/\gamma$ sets how fast they diffuse

In [25]:
Brownian = hoomd.md.methods.Brownian(filter=hoomd.filter.All(), kT=kT) # Brownian dynamics for all particles at temperature kT

In [26]:
Brownian.gamma.default = 1.0 # drag coefficient for every particle type

In [27]:
integrator.methods.append(Brownian) # use Brownian dynamics to move the particles

### Interactions
- Lennard-Jones: $U(r) = 4\epsilon\left[(\sigma/r)^{12} - (\sigma/r)^6\right]$, strongly repulsive below $r \approx \sigma$ (particles can't overlap) and weakly attractive further out; $\epsilon$ sets the strength
- We ignore interactions beyond a cutoff `r_cut`, and the neighbour list keeps track of which pairs are within it so HOOMD doesn't check every pair every step

In [28]:
cell = hoomd.md.nlist.Cell(buffer=0.4) # neighbour list: finds which particles are close enough to interact

In [29]:
lj = hoomd.md.pair.LJ( nlist=cell, mode='shift') # Lennard-Jones interaction, shifted so the energy is 0 at the cutoff
lj.params.default = dict(epsilon=1.0, sigma=1.0) # interaction strength and particle size
lj.r_cut.default = 2.5 # no interaction beyond 2.5 sigma
integrator.forces.append(lj) # add the LJ force to the integrator

In [30]:
# delete any old trajectory file
trajectory_filename = "trajectory.gsd"
if os.path.exists(trajectory_filename):
    os.remove(trajectory_filename)

In [31]:
simulation.operations.integrator = integrator # attach the integrator to the simulation

In [32]:
# save the trajectory to file
gsd_writer = hoomd.write.GSD(filename=trajectory_filename, 
                            trigger=hoomd.trigger.Periodic(period=100), # save a frame every 100 steps
                            mode="wb", # overwrite any existing file
                            filter=hoomd.filter.All(), # save all particles
                            dynamic=['property', 'particles/image']) # save positions and box crossings every frame
simulation.operations.writers.append(gsd_writer) # attach the writer to the simulation

In [33]:
simulation.run(100000) # run for 100,000 steps
gsd_writer.flush() # write any frames still held in memory to the file


### Measuring diffusion
- The mean squared displacement $\langle|\mathbf{r}(t)-\mathbf{r}(0)|^2\rangle$ of Brownian particles grows as $4D_tt$ in 2D
- Undo the periodic wrapping first using the box crossings saved in the trajectory (`particles/image`); the `freud` package (`freud.msd`) does the calculation